In [103]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
os.chdir('C:/Users/edwin/OneDrive/Documents/GitHub/Superconductor') # Change to your own directory
# os.chdir('') 


In [ ]:
data = pd.read_csv('data/train.csv')
element = pd.read_csv('data/unique_m.csv')
element = element.drop(columns=['critical_temp']) 

elementsum = element.drop(columns=['material']).sum(axis = 1)
element_norm = element.copy()
# Normalizing elements, rounding to avoid floating points
element_norm.loc[:,element_norm.columns != 'material'] = element_norm.loc[:,element_norm.columns != 'material'].div(elementsum, axis = 0).round(8) 

# Concatenating both datasets by columns (They have the same rows)
superconductor = pd.concat([data, element], axis = 1)
superconductor = superconductor.drop(columns = ['He', 'Ne', 'Ar', 'Kr', 'Xe', 'Pm', 'Po', 'At', 'Rn']) # dropping unused elements

# doing the same for normalized element data
superconductor_norm = pd.concat([data, element_norm], axis = 1) 
superconductor_norm = superconductor_norm.drop(columns = ['He', 'Ne', 'Ar', 'Kr', 'Xe', 'Pm', 'Po', 'At', 'Rn']) 

In [ ]:
superconductor['material'].duplicated().sum() # there are many duplicated compounds and different critical temperatures for the same compound. 

5721

In [109]:
# Get the element columns that remain in the dataset
element_cols = [
    col for col in element.columns
    if col != 'material' and col in superconductor_norm.columns
]

# Group observations by their normalized elemental composition
groups = superconductor_norm[element_cols].apply(tuple, axis=1).factorize()[0]

In [115]:
# Split into training and test sets
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=1
)

train_idx, temp_idx = next(
    gss.split(superconductor_norm, groups=groups)
)

# Second split for validation and test set
gss2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=1
)

val_pos, test_pos = next(
    gss2.split(
        superconductor_norm.iloc[temp_idx],
        groups=groups[temp_idx]
    )
)

val_idx = temp_idx[val_pos]
test_idx = temp_idx[test_pos]

In [117]:
# Raw dataset
raw_train = superconductor.iloc[train_idx].copy()
raw_val = superconductor.iloc[val_idx].copy()
raw_test = superconductor.iloc[test_idx].copy()

# Normalized dataset
norm_train = superconductor_norm.iloc[train_idx].copy()
norm_val = superconductor_norm.iloc[val_idx].copy()
norm_test = superconductor_norm.iloc[test_idx].copy()

In [121]:
# Verifying dataset 

# Verify no overlapping compositions
assert set(groups[train_idx]).isdisjoint(groups[val_idx])
assert set(groups[train_idx]).isdisjoint(groups[test_idx])
assert set(groups[val_idx]).isdisjoint(groups[test_idx])

# Verify all observations were assigned
assert len(train_idx) + len(val_idx) + len(test_idx) == len(superconductor)


In [123]:
# Split response and predictors

# Raw dataset
X_train_raw = raw_train.drop(columns=['critical_temp', 'material'])
X_val_raw = raw_val.drop(columns=['critical_temp', 'material'])
X_test_raw = raw_test.drop(columns=['critical_temp', 'material'])

y_train = raw_train['critical_temp']
y_val = raw_val['critical_temp']
y_test = raw_test['critical_temp']

# Normalized dataset
X_train_norm = norm_train.drop(columns=['critical_temp', 'material'])
X_val_norm = norm_val.drop(columns=['critical_temp', 'material'])
X_test_norm = norm_test.drop(columns=['critical_temp', 'material'])

In [125]:
# Exporting datasets

# Raw element features
X_train_raw.to_csv('data/processed/X_train_raw.csv', index=False)
X_val_raw.to_csv('data/processed/X_val_raw.csv', index=False)
X_test_raw.to_csv('data/processed/X_test_raw.csv', index=False)

# Normalized element features
X_train_norm.to_csv('data/processed/X_train_norm.csv', index=False)
X_val_norm.to_csv('data/processed/X_val_norm.csv', index=False)
X_test_norm.to_csv('data/processed/X_test_norm.csv', index=False)

# Targets (shared by both datasets)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_val.to_csv('data/processed/y_val.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)